# 03 (4B) - Select pairs of attempts
Same as notebook 03, but on the attempts written by the 4B model (`chains_qwen4b.json`). Output: `datasets/review_pairs_qwen4b.json`.

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

In [2]:
path=Path('../datasets/chains_qwen4b.json') # chains written by the 4B model (notebook 02 with model_tag='qwen4b')

with path.open('r',encoding='utf-8') as file:
    dataset=json.load(file)

print(len(dataset))

7498


In [3]:
dataset[0]['chains'][0]

{'text': ' Let the initial number of blue marbles be 8x and the initial number of yellow marbles be 5x, for some positive integer x. After removing 12 blue marbles, the number of blue marbles becomes 8x - 12. After adding 21 yellow marbles, the number of yellow marbles becomes 5x + 21. The new ratio of blue marbles to yellow marbles is 1:3, which can be written as: \\[ \\frac{8x - 12}{5x + 21} = \\frac{1}{3} \\] Cross-multiplying gives: \\[ 3(8x - 12) = 5x + 21 \\] Expanding and simplifying: \\[ 24x - 36 = 5x + 21 \\] \\[ 24x - 5x = 21 + 36 \\] \\[ 19x = 57 \\] \\[ x = 3 \\] Therefore, the initial number of blue marbles is: \\[ 8x = 8 \\times 3 = 24 \\] \\[ \\boxed{24} \\]',
 'pred': '24',
 'ok': True,
 'finished': True,
 'n_tokens': 265}

In [4]:
def is_usable(chain: dict)->bool:
    return chain['finished'] and chain['n_tokens']<=700

In [5]:
random.seed(0)

pairs=[]
agree_candidates=[]

for i in dataset:
    correct=[c for c in i['chains'] if c['ok'] and is_usable(c)]
    wrong=[c for c in i['chains'] if (not c['ok']) and is_usable(c)]

    if correct and wrong:
        right_chain=random.choice(correct)
        wrong_chain=random.choice(wrong)
        if random.random()<0.5:
            pair_type='correction'
            answers=[wrong_chain,right_chain]
        else:
            pair_type='keep'
            answers=[right_chain,wrong_chain]
    elif len(correct)>=2:
        pair_type='agree'
        answers=random.sample(correct,2)
    else:
        continue

    record={
        'id': i['id'],
        'level': i['level'],
        'type': pair_type,
        'problem': i['problem'],
        'gold': i['gold'],
        'answer_1': answers[0]['text'].strip(),
        'answer_2': answers[1]['text'].strip(),
        'pred_1': answers[0]['pred'],
        'pred_2': answers[1]['pred'],
        'other_correct': [c['text'].strip() for c in correct],
    }

    if pair_type=='agree':
        agree_candidates.append(record)
    else:
        pairs.append(record)

print(len(pairs),len(agree_candidates))

1677 3581


In [6]:
agree_pairs=random.sample(agree_candidates,1000)
pairs=pairs+agree_pairs
random.shuffle(pairs)

print(Counter(p['type'] for p in pairs))

Counter({'agree': 1000, 'keep': 839, 'correction': 838})


In [7]:
Counter(p['level'] for p in pairs)

Counter({'Level 4': 666,
         'Level 3': 663,
         'Level 5': 632,
         'Level 2': 517,
         'Level 1': 198,
         'Level ?': 1})

In [8]:
assert all(p['pred_1']!=None and p['pred_2']!=None for p in pairs)

In [9]:
pairs[0]

{'id': 'math-7401',
 'level': 'Level 4',
 'type': 'agree',
 'problem': 'Given that $x$ is real and $x^3+\\frac{1}{x^3}=52$, find $x+\\frac{1}{x}$.',
 'gold': '4',
 'answer_1': 'We are given that \\(x\\) is a real number satisfying \\[x^3+\\frac{1}{x^3}=52.\\] Let \\(y = x + \\frac{1}{x}\\). Then we can express \\(x^3+\\frac{1}{x^3}\\) in terms of \\(y\\). First, note that: \\[y = x + \\frac{1}{x} \\implies y^2 = x^2 + 2 + \\frac{1}{x^2} \\implies x^2 + \\frac{1}{x^2} = y^2 - 2.\\] Squaring both sides of the equation for \\(y^2\\), we get: \\[y^3 = x^3 + 3x + \\frac{3}{x} + \\frac{1}{x^3}.\\] This can be rewritten as: \\[y^3 = \\left(x^3 + \\frac{1}{x^3}\\right) + 3\\left(x + \\frac{1}{x}\\right).\\] Substituting the given value \\(x^3 + \\frac{1}{x^3} = 52\\) and \\(y = x + \\frac{1}{x}\\), we get: \\[y^3 = 52 + 3y.\\] Rearranging this equation, we have: \\[y^3 - 3y - 52 = 0.\\] Now, we need to solve the cubic equation \\(y^3 - 3y - 52 = 0\\). We can try to find rational roots using th

In [10]:
savepath=Path('../datasets/review_pairs_qwen4b.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(pairs,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)

Saved: ../datasets/review_pairs_qwen4b.json
